# Minimum Teacher Investment Eval

**The question**: How little teacher training produces how much student benefit?

**Design**:
- Data split: Train (70%) / Val (15%) / Test (15%) — strict partition
- Teacher trains on Train only, for {50, 100, 200, 500, 1000, 2000} steps
- Each teacher produces 128 bytes of spectra (Tier 1 only — no directions)
- Student trains on Train with Marathon modulation, evaluated on Test
- No data leakage by construction: spectra never touch Test partition

**The output**: one curve showing teacher investment vs student speedup.

In [ ]:
# Cell 1: Setup + create strict data partition
import os, subprocess, numpy as np, shutil
if os.path.exists('/content/nanogpt-prism'):
    subprocess.run(['rm', '-rf', '/content/nanogpt-prism'])
!git clone https://github.com/realityinspector/nanogpt-prism.git /content/nanogpt-prism
%cd /content/nanogpt-prism
!pip install -q transformers tiktoken datasets
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
!python data/shakespeare_char/prepare.py 2>/dev/null

# Create strict 70/15/15 partition
data = np.array(np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r'))
val_orig = np.array(np.memmap('data/shakespeare_char/val.bin', dtype=np.uint16, mode='r'))

# Combine all data, then re-split
all_data = np.concatenate([data, val_orig])
np.random.seed(42)
np.random.shuffle(all_data)  # shuffle at token level to break any ordering bias

n = len(all_data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = all_data[:train_end].astype(np.uint16)
val_data = all_data[train_end:val_end].astype(np.uint16)
test_data = all_data[val_end:].astype(np.uint16)

# Save as eval dataset (student trains + evals on this)
d = 'data/shakespeare_eval'
os.makedirs(d, exist_ok=True)
train_data.tofile(os.path.join(d, 'train.bin'))
test_data.tofile(os.path.join(d, 'val.bin'))  # nanoGPT calls it val.bin but it's our test set
shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d, 'meta.pkl'))

# Save teacher dataset (same train, but val is the actual val for early stopping)
d2 = 'data/shakespeare_teacher'
os.makedirs(d2, exist_ok=True)
train_data.tofile(os.path.join(d2, 'train.bin'))
val_data.tofile(os.path.join(d2, 'val.bin'))
shutil.copy('data/shakespeare_char/meta.pkl', os.path.join(d2, 'meta.pkl'))

print(f'Train: {len(train_data):,} tokens')
print(f'Val:   {len(val_data):,} tokens')
print(f'Test:  {len(test_data):,} tokens (never seen by teacher or student during training)')
print('Partitions ready.')

In [ ]:
# Cell 2: Train teachers at different investment levels + extract spectra
import subprocess, os

teacher_steps = [50, 100, 200, 500, 1000, 2000]

for steps in teacher_steps:
    cache = f'.prism_cache/teacher_{steps}'
    if os.path.exists(f'{cache}/spectra.json'):
        print(f'  [{steps} steps] Cached.')
        continue
    
    print(f'\n  Training teacher ({steps} steps)...')
    result = subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--dataset=shakespeare_teacher',
        f'--max_iters={steps}',
        f'--eval_interval={steps}',
        '--eval_iters=50',
        f'--log_interval={steps}',
        f'--out_dir=out-teacher-{steps}',
        '--always_save_checkpoint=True',
        '--compile=False',
        '--prism_init=False',
        '--wandb_log=False',
    ], capture_output=True, text=True, timeout=600)
    
    # Show teacher's final loss
    import re
    for line in result.stdout.split('\n'):
        m = re.search(r'step \d+: train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            print(f'    Teacher val loss: {m.group(2)}')
    
    if result.returncode != 0:
        print(f'    ERROR: {result.stderr[-300:]}')
        continue
    
    # Extract spectra (128 bytes)
    subprocess.run([
        'python', 'prism_extract.py',
        '--ckpt', f'out-teacher-{steps}/ckpt.pt',
        '--out', cache,
    ], capture_output=True, timeout=120)
    
    # Show spectra file size
    spec_size = os.path.getsize(f'{cache}/spectra.json')
    print(f'    Spectra: {spec_size} bytes')

print('\nAll teachers trained and extracted.')

In [ ]:
# Cell 3: Run baseline + all students (spectral-only Marathon, no directions)
import subprocess, time, re, json, os

STEPS = 5000
EVAL = 100
DATASET = 'shakespeare_eval'  # trains on train partition, evals on TEST partition

configs = [('baseline', ['--prism_init=False'])]

# Spectral-only (Tier 1: 128 bytes, no directions) with Marathon mod
for teacher_steps in [50, 100, 200, 500, 1000, 2000]:
    configs.append((f'spec_{teacher_steps}', [
        '--prism_init=True', '--prism_align=0',  # align=0 means spectral only, no directions
        f'--prism_spectra=.prism_cache/teacher_{teacher_steps}/spectra.json',
        '--learning_rate=5e-4', '--warmup_iters=50',
        '--prism_mod=0.01', '--prism_mod_decay=0.9999',
    ]))

# Also test best teacher WITH directions (for comparison)
configs.append(('full_2000', [
    '--prism_init=True', '--prism_align=0.75',
    '--prism_spectra=.prism_cache/teacher_2000/spectra.json',
    '--prism_directions=.prism_cache/teacher_2000/directions.pt',
    '--learning_rate=5e-4', '--warmup_iters=50',
    '--prism_mod=0.01', '--prism_mod_decay=0.9999',
]))

all_curves = {}

for name, extra in configs:
    log_path = f'/content/minteach_{name}.txt'
    if os.path.exists(log_path):
        print(f'[{name}] SKIP')
        with open(log_path) as f:
            stdout = f.read()
    else:
        print(f'\n  {name}...')
        t0 = time.time()
        result = subprocess.run(
            ['python', 'train.py', 'config/train_shakespeare_char.py',
             f'--dataset={DATASET}',
             f'--max_iters={STEPS}', f'--eval_interval={EVAL}',
             '--eval_iters=50', '--log_interval=250',
             f'--out_dir=out-minteach-{name}',
             '--wandb_log=False', '--compile=False'] + extra,
            capture_output=True, text=True, timeout=1200
        )
        stdout = result.stdout
        with open(log_path, 'w') as f:
            f.write(stdout)
        if result.returncode != 0:
            print(f'    ERROR: {result.stderr[-300:]}')
        print(f'    Wall: {time.time()-t0:.0f}s')

    val = {}
    for line in stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m:
            val[int(m.group(1))] = float(m.group(3))
    all_curves[name] = val
    if val:
        best = min(val.values())
        print(f'  [{name}] best: {best:.4f}')

with open('/content/minteach_curves.json', 'w') as f:
    json.dump({k: {str(s): v for s, v in c.items()} for k, c in all_curves.items()}, f, indent=2)
print('\nAll saved.')

In [ ]:
# Cell 4: The headline number
import json

curves = {k: {int(s): v for s, v in c.items()}
          for k, c in json.load(open('/content/minteach_curves.json')).items()}

bb = min(curves['baseline'].values())
bs = min(curves['baseline'], key=curves['baseline'].get)

print('='*70)
print('  MINIMUM TEACHER INVESTMENT EVAL')
print('='*70)
print(f'  Strict partition: Train/Val/Test never overlap.')
print(f'  All eval on held-out TEST set.')
print(f'  Spectral-only (128 bytes, no directions) with Marathon mod.')
print(f'\n  Baseline best: {bb:.4f} at step {bs}')

print(f'\n  {"Teacher":>10s}  {"128 bytes":>10s}  {"Best test":>10s}  {"@step":>6s}  {"Hit base":>9s}  {"Speedup":>8s}  {"@5000":>8s}')
print(f'  {"-"*70}')

# Baseline
print(f'  {"(none)":>10s}  {"—":>10s}  {bb:>10.4f}  {bs:>6d}  {bs:>9d}  {"1.0x":>8s}  {curves["baseline"].get(5000,0):>8.4f}')

for teacher_steps in [50, 100, 200, 500, 1000, 2000]:
    name = f'spec_{teacher_steps}'
    c = curves.get(name, {})
    if not c: continue
    best = min(c.values())
    best_s = min(c, key=c.get)
    hit = None
    for s in sorted(c.keys()):
        if c[s] <= bb:
            hit = s
            break
    spd = f'{bs/hit:.1f}x' if hit else '—'
    hit_str = str(hit) if hit else 'never'
    at_5k = c.get(5000, c.get(max(c.keys()), 0))
    print(f'  {str(teacher_steps)+" steps":>10s}  {"128 B":>10s}  {best:>10.4f}  {best_s:>6d}  {hit_str:>9s}  {spd:>8s}  {at_5k:>8.4f}')

# Full (with directions)
c = curves.get('full_2000', {})
if c:
    best = min(c.values())
    best_s = min(c, key=c.get)
    hit = None
    for s in sorted(c.keys()):
        if c[s] <= bb:
            hit = s
            break
    spd = f'{bs/hit:.1f}x' if hit else '—'
    hit_str = str(hit) if hit else 'never'
    at_5k = c.get(5000, c.get(max(c.keys()), 0))
    print(f'  {"2000+dirs":>10s}  {"~500MB":>10s}  {best:>10.4f}  {best_s:>6d}  {hit_str:>9s}  {spd:>8s}  {at_5k:>8.4f}')

print(f'\n  The headline: how many teacher steps (seconds of compute) produce')
print(f'  a useful 128-byte spectral fingerprint?')

In [ ]:
# Cell 5: Download
from google.colab import files
files.download('/content/minteach_curves.json')